In [7]:
%pip install datasets

import os
import pandas as pd
import torch
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset

# Optional HF dataset iterator for efficient batched inference
try:
    from datasets import Dataset
except ImportError as exc:
    raise ImportError(
        "Please install 'datasets' in your environment before running this notebook."
    ) from exc

# Config
INPUT_FILE = "/kaggle/input/new-processed/Processed_Reviews.csv"
INPUT_FILE = "Processed_Reviews.csv"
BATCH_SIZE = 32
MAX_TEXT_LEN = 256
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"
MODEL_REVISION = "daefdd1f6ae931839bce4d0f3db0a1a4265cd50f"
PIPELINE_DEVICE = 0 if torch.cuda.is_available() else -1


def build_output_path(input_file):
    input_abs = os.path.abspath(input_file)
    input_dir = os.path.dirname(input_abs)
    base_name = os.path.splitext(os.path.basename(input_abs))[0]
    out_name = f"{base_name}_with_Sentiment.csv"

    if os.path.isdir(input_dir) and os.access(input_dir, os.W_OK):
        return os.path.join(input_dir, out_name)

    for candidate_dir in ["/kaggle/working", os.getcwd()]:
        try:
            os.makedirs(candidate_dir, exist_ok=True)
            if os.access(candidate_dir, os.W_OK):
                return os.path.join(candidate_dir, out_name)
        except Exception:
            pass

    return out_name


OUTPUT_FILE = build_output_path(INPUT_FILE)

# Load data
df = pd.read_csv(INPUT_FILE)

# Build combined text from Title + Text
if "Title" not in df.columns and "Text" not in df.columns:
    raise ValueError("Input must contain at least one of: 'Title' or 'Text'.")

title = df["Title"].fillna("").astype(str) if "Title" in df.columns else pd.Series([""] * len(df))
text = df["Text"].fillna("").astype(str) if "Text" in df.columns else pd.Series([""] * len(df))
combined_text = (title + " " + text).str.strip()

# Prepare valid rows
valid_indices = [i for i, t in enumerate(combined_text.tolist()) if isinstance(t, str) and t.strip()]
valid_texts = [combined_text.iloc[i] for i in valid_indices]

label_map = {"LABEL_0": "NEGATIVE", "LABEL_1": "NEUTRAL", "LABEL_2": "POSITIVE"}
predicted = [None] * len(combined_text)

# Load model with pinned revision for reproducibility
sentiment_pipe = pipeline(
    "text-classification",
    model=MODEL_NAME,
    revision=MODEL_REVISION,
    device=PIPELINE_DEVICE,
)

if valid_texts:
    infer_ds = Dataset.from_dict({"text": valid_texts})
    outputs = sentiment_pipe(
        KeyDataset(infer_ds, "text"),
        truncation=True,
        max_length=MAX_TEXT_LEN,
        batch_size=BATCH_SIZE,
    )

    for idx, out in zip(valid_indices, outputs):
        raw_label = str(out.get("label", "")).strip().upper()
        predicted[idx] = label_map.get(raw_label, raw_label if raw_label else None)

# Append only one new column and save
df["Sentiment"] = predicted
df.to_csv(OUTPUT_FILE, index=False)

print("Saved output:", OUTPUT_FILE)
print("Final shape:", df.shape)
print(df[["Sentiment"]].head())

  Using cached xxhash-3.6.0-cp311-cp311-win_amd64.whl.metadata (13 kB)
     ---------------------------------------- 0.0/82.2 kB ? eta -:--:--
     -------------- ------------------------- 30.7/82.2 kB 1.3 MB/s eta 0:00:01
     ---------------------------------------- 82.2/82.2 kB 1.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   -------- ------------------------------- 112.6/527.0 kB 6.4 MB/s eta 0:00:01
   ---------- ----------------------------- 143.4/527.0 kB 2.1 MB/s eta 0:00:01
   ----------------- ---------------------- 235.5/527.0 kB 2.4 MB/s eta 0:00:01
   --------------------------- ------------ 358.4/527.0 kB 2.8 MB/s eta 0:00:01
   --------------------------------- ------ 440.3/527.0 kB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 527.0/527.0 kB 2.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/120.0 kB ? eta -:--:--
   ------------------------------ --------- 92.2/120.0 kB 2.6 MB/s eta 0:00


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14701.46it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 